## Recommendations with MovieTweetings: Collaborative Filtering

One of the most popular methods for making recommendations is **collaborative filtering**.  In collaborative filtering, you are using the collaboration of user-item recommendations to assist in making new recommendations.  

In this notebook, you will be working on performing **collaborative filtering**.  There are two main methods for performing collaborative filtering:

1. **User-based collaborative filtering:** In this type of recommendation, users related to the user you would like to make recommendations for are used to create a recommendation.

2. **Item-based collaborative filtering:** In this type of recommendation, first you need to find the items that are most related to each other item (based on similar ratings).  Then you can use the ratings of an individual on those similar items to understand if a user will like the new item.

In this notebook you will be implementing **user-based collaborative filtering**.  However, it is easy to extend this approach to make recommendations using **item-based collaborative filtering**.  First, let's read in our data and necessary libraries.

**NOTE**: Because of the size of the datasets, some of your code cells here will take a while to execute, so be patient!

In [1]:
# NumPy auxilia nas operações numéricas; pandas manipula as tabelas.
import numpy as np
import pandas as pd

# Carrega o catálogo de filmes e o histórico de avaliações.
movies = pd.read_csv('data/movies_clean.csv')
reviews = pd.read_csv('data/reviews_clean.csv')

# Remove a coluna de índice criada durante a exportação dos CSVs.
del movies['Unnamed: 0']
del reviews['Unnamed: 0']

# Exibe uma amostra para conferir as colunas e os tipos carregados.
print(reviews.head())

   user_id  movie_id  rating   timestamp                 date
0        1    114508       8  1381006850  2013-10-05 21:00:50
1        2    208092       5  1586466072  2020-04-09 21:01:12
2        2    358273       9  1579057827  2020-01-15 03:10:27
3        2  10039344       5  1578603053  2020-01-09 20:50:53
4        2   6751668       9  1578955697  2020-01-13 22:48:17


In the above matrix, you can see that **User 1** and **User 2** both used **Item 1**, and **User 2**, **User 3**, and **User 4** all used **Item 2**.  However, there are also a large number of missing values in the matrix for users who haven't used a particular item.  A matrix with many missing values (like the one above) is considered **sparse**.

Our first goal for this notebook is to create the above matrix with the **reviews** dataset.  However, instead of 1 values in each cell, you should have the actual rating.  

The users will indicate the rows, and the movies will exist across the columns. To create the user-item matrix, we only need the first three columns of the **reviews** dataframe, which you can see by running the cell below.

In [2]:
# Mantém apenas as três informações necessárias para montar a matriz.
user_items = reviews[['user_id', 'movie_id', 'rating']]
user_items.head()

,user_id,movie_id,rating
0,1,114508,8
1,2,208092,5
2,2,358273,9
3,2,10039344,5
4,2,6751668,9


### Creating the User-Item Matrix

In order to create the user-items matrix (like the one above), I personally started by using a [pivot table](https://pandas.pydata.org/pandas-docs/stable/generated/pandas.pivot_table.html). 

However, I quickly ran into a memory error (a common theme throughout this notebook).  I will help you navigate around many of the errors I had, and achieve useful collaborative filtering results! 

_____

`1.` Create a matrix where the users are the rows, the movies are the columns, and the ratings exist in each cell, or a NaN exists in cells where a user hasn't rated a particular movie. If you get a memory error (like I did), [this link here](https://stackoverflow.com/questions/39648991/pandas-dataframe-pivot-memory-error) might help you!

In [3]:
# A matriz é muito esparsa; CSR armazena somente as avaliações existentes.
from scipy.sparse import csr_matrix

# Ordenar os IDs cria uma posição estável para cada usuário e filme.
user_ids = np.sort(user_items['user_id'].unique())
movie_ids = np.sort(user_items['movie_id'].unique())

# Converte os IDs originais em coordenadas inteiras de linha e coluna.
user_positions = pd.Categorical(user_items['user_id'], categories=user_ids).codes
movie_positions = pd.Categorical(user_items['movie_id'], categories=movie_ids).codes

# Posiciona cada rating na coordenada (usuário, filme) correspondente.
rating_matrix = csr_matrix(
    (user_items['rating'].to_numpy(), (user_positions, movie_positions)),
    shape=(len(user_ids), len(movie_ids))
)

# Recupera os rótulos dos IDs sem transformar a matriz em um array denso.
user_by_movie = pd.DataFrame.sparse.from_spmatrix(
    rating_matrix, index=user_ids, columns=movie_ids
)
# Na primeira etapa, NaN indica que o usuário não avaliou o filme.
user_by_movie = user_by_movie.astype(
    pd.SparseDtype('float64', fill_value=np.nan)
)
# Nomeia os eixos para deixar explícito o significado da matriz.
user_by_movie.index.name = 'user_id'
user_by_movie.columns.name = 'movie_id'

Check your results below to make sure your matrix is ready for the upcoming sections.

In [4]:
# Confirma que nenhum usuário foi perdido durante a construção da matriz.
assert reviews.user_id.nunique() == user_by_movie.shape[0], "Oh no! Your matrix should have {} rows, and yours has {}!".format(reviews.user_id.nunique(), user_by_movie.shape[0])
print("Looks like you are all set! Proceed!")

Looks like you are all set! Proceed!


`2.` For a first iteration, we will update the user-movie matrix imagine we are only interested in which movies a user has viewed.  Therefore, update your solution to question 1 to have a `0` for any movie a user has not rated and a `1` for any movie a user has rated.

In [5]:
# .gt(0) marca como True cada avaliação positiva e False cada ausência.
# int8 converte True/False em 1/0 usando apenas um byte por valor armazenado.
user_by_movie = (
    user_by_movie
    .gt(0)
    .astype(pd.SparseDtype('int8', fill_value=0))
)

`3.` A common similarity metric is `cosine_similarity`.  Complete the function below using this metric to determine which users are most similar to one another.

In [6]:
# A similaridade de cosseno mede o alinhamento entre os vetores de interações.
from sklearn.metrics.pairwise import cosine_similarity

def find_similar_users(user_id, user_item, include_similarity=False):
    """
    INPUT:
    user_id - (int) a user_id
    user_item - (pandas dataframe) matrix of users by movies:
                1's when a user has interacted with a movie, 0 otherwise
    include_similarity - (bool) whether to include the similarity in the output

    OUTPUT:
    similar_users - (list) an ordered list where the closest users (largest cosine similarity)
                    are listed first

    Description:
    Computes the cosine similarity between the selected user and every other user.
    Returns an ordered list of user ids. If include_similarity is True, returns a list of lists
    where the first element is the user id and the second the similarity.

    """
    # Falha cedo com uma mensagem clara quando o usuário não existe.
    if user_id not in user_item.index:
        raise KeyError(f'User {user_id} is not present in the user-item matrix.')

    # O scikit-learn calcula o cosseno diretamente sobre a matriz CSR esparsa.
    user_matrix = user_item.sparse.to_coo().tocsr()
    # Localiza a linha do usuário consultado sem assumir que ID e posição são iguais.
    user_position = user_item.index.get_loc(user_id)
    # Compara apenas a linha alvo com todas as linhas da matriz.
    similarities = cosine_similarity(
        user_matrix[user_position], user_matrix
    ).ravel()

    # Associa cada similaridade ao respectivo user_id.
    similarity_scores = pd.DataFrame({
        'user_id': user_item.index.to_numpy(),
        'similarity': similarities
    })
    # Remove o próprio usuário e ordena do mais para o menos semelhante.
    # O user_id funciona como critério determinístico para empates.
    similarity_scores = (
        similarity_scores[similarity_scores['user_id'] != user_id]
        .sort_values(
            ['similarity', 'user_id'],
            ascending=[False, True],
            ignore_index=True
        )
    )

    # Opcionalmente retorna cada ID acompanhado de sua similaridade.
    if include_similarity:
        return [
            [int(row.user_id), float(row.similarity)]
            for row in similarity_scores.itertuples(index=False)
        ]

    # No uso padrão, devolve somente os IDs já na ordem de proximidade.
    return similarity_scores['user_id'].astype(int).tolist()

`4.` Using your new function, who are the top 10 similar users to user `3508`?

In [7]:
# Solicita as similaridades e seleciona somente os dez vizinhos mais próximos.
top_similar_users = find_similar_users(
    3508, user_by_movie, include_similarity=True
)[:10]
# Extrai apenas os IDs para consultar suas interações na matriz.
users = [neighbor_id for neighbor_id, _ in top_similar_users]

pd.DataFrame(top_similar_users, columns=['user_id', 'cosine_similarity'])

,user_id,cosine_similarity
0,2544,0.816497
1,5544,0.816497
2,27,0.577350
3,1254,0.577350
4,2591,0.577350
5,3158,0.577350
6,5808,0.577350
7,7598,0.577350
8,100,0.462910
9,6376,0.436436


`5.` Determine what movies you would recommend to user `3508` by finding movies they haven't rated that the top 10 users from the previous question have rated.

In [8]:
# Seleciona as colunas marcadas com 1 na linha do usuário 3508.
movies_seen = user_by_movie.columns[
    user_by_movie.loc[3508].to_numpy(dtype=np.int8) > 0
]

# Recorta as linhas dos vizinhos sem densificar toda a matriz.
neighbor_matrix = user_by_movie.loc[users].sparse.to_coo().tocsr()
# A soma por coluna informa quantos vizinhos interagiram com cada filme.
neighbor_counts = np.asarray(neighbor_matrix.sum(axis=0)).ravel()

# Converte as contagens em uma Series indexada por movie_id.
recommendation_scores = pd.Series(
    neighbor_counts, index=user_by_movie.columns, name='similar_user_count'
)
# Exclui filmes já vistos, pois não devem ser recomendados novamente.
recommendation_scores = recommendation_scores.drop(index=movies_seen)
# Mantém candidatos vistos por pelo menos um vizinho e prioriza os mais populares.
recommendation_scores = (
    recommendation_scores[recommendation_scores > 0]
    .sort_values(ascending=False, kind='stable')
)

In [9]:
# Transforma os scores em tabela e acrescenta o título legível de cada filme.
recommendations = (
    recommendation_scores
    .rename_axis('movie_id')
    .reset_index()
    .merge(movies[['movie_id', 'movie']], on='movie_id', how='left')
    [['movie_id', 'movie', 'similar_user_count']]
)
# Contagens são números inteiros; a conversão deixa o resultado mais claro.
recommendations['similar_user_count'] = (
    recommendations['similar_user_count'].astype(int)
)

# Garante que nenhum filme já avaliado entrou acidentalmente nas recomendações.
assert set(recommendations['movie_id']).isdisjoint(movies_seen)
# Exibe as recomendações mais fortes primeiro.
recommendations.head(20)

,movie_id,movie,similar_user_count
0,101414,Beauty and the Beast (1991),1
1,110478,Maverick (1994),1
2,313443,Out of Time (2003),1
3,482571,The Prestige (2006),1
4,816711,World War Z (2013),1
5,1210819,The Lone Ranger (2013),1
6,1343092,The Great Gatsby (2013),1
7,1392214,Prisoners (2013),1
8,1490017,The Lego Movie (2014),1
9,1690953,Despicable Me 2 (2013),1
